### Persistence in Graphs

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

load_dotenv()
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

chat_model = AzureChatOpenAI(azure_endpoint=azure_endpoint,
    api_key=api_key,
    api_version=api_version,
    deployment_name=deployment_name)

c:\Users\HarshVardhanSingh\anaconda3\envs\demo_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [3]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = chat_model.invoke(prompt).content
    return {'joke': response}


def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = chat_model.invoke(prompt).content
    return {'explanation': response}


In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)


# We need to apply a checkpointer to save the state of the workflow
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [6]:
# During invocation we need to pass threads
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Stephen Curry'}, config=config1)

{'topic': 'Stephen Curry',
 'joke': 'Why did Stephen Curry bring a ladder to the game?\n\nBecause he wanted to shoot for the stars!',
 'explanation': 'The joke plays on a couple of different ideas. First, Stephen Curry is a professional basketball player known for his exceptional shooting skills, particularly his ability to make three-point shots. The phrase "shoot for the stars" is a common expression that means to aim for ambitious goals or high achievements. \n\nIn the context of the joke, the ladder adds a humorous twist. Ladders are typically used to reach high places, so bringing one to a basketball game suggests a literal interpretation of the phrase. The punchline combines Curry\'s basketball prowess with the metaphorical idea of aspiring to great heights, creating a playful and lighthearted image of him literally trying to reach the stars. \n\nOverall, the humor arises from the wordplay and the absurdity of the situation, as it\'s not typical for a basketball player to need a 

In [7]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Stephen Curry', 'joke': 'Why did Stephen Curry bring a ladder to the game?\n\nBecause he wanted to shoot for the stars!', 'explanation': 'The joke plays on a couple of different ideas. First, Stephen Curry is a professional basketball player known for his exceptional shooting skills, particularly his ability to make three-point shots. The phrase "shoot for the stars" is a common expression that means to aim for ambitious goals or high achievements. \n\nIn the context of the joke, the ladder adds a humorous twist. Ladders are typically used to reach high places, so bringing one to a basketball game suggests a literal interpretation of the phrase. The punchline combines Curry\'s basketball prowess with the metaphorical idea of aspiring to great heights, creating a playful and lighthearted image of him literally trying to reach the stars. \n\nOverall, the humor arises from the wordplay and the absurdity of the situation, as it\'s not typical for a basketbal

In [ ]:
# The values are stored in each state, over the period of time.
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Stephen Curry', 'joke': 'Why did Stephen Curry bring a ladder to the game?\n\nBecause he wanted to shoot for the stars!', 'explanation': 'The joke plays on a couple of different ideas. First, Stephen Curry is a professional basketball player known for his exceptional shooting skills, particularly his ability to make three-point shots. The phrase "shoot for the stars" is a common expression that means to aim for ambitious goals or high achievements. \n\nIn the context of the joke, the ladder adds a humorous twist. Ladders are typically used to reach high places, so bringing one to a basketball game suggests a literal interpretation of the phrase. The punchline combines Curry\'s basketball prowess with the metaphorical idea of aspiring to great heights, creating a playful and lighthearted image of him literally trying to reach the stars. \n\nOverall, the humor arises from the wordplay and the absurdity of the situation, as it\'s not typical for a basketba

In [9]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Lebron James'}, config=config2)

{'topic': 'Lebron James',
 'joke': 'Why did LeBron James bring a ladder to the game?\n\nBecause he heard the championship was up for grabs!',
 'explanation': 'This joke plays on a couple of puns related to basketball and the concept of "grabs." \n\n1. **Ladder in Basketball**: In basketball, a ladder is often used by players or staff to reach the net for tasks like cutting down the net after winning a championship. This creates a visual image of LeBron James literally bringing a ladder to help him achieve victory.\n\n2. **"Up for Grabs"**: The phrase "up for grabs" is an idiom meaning that something is available to be taken or won. In this context, it refers to the championship being contested, implying that the title is open for any team to claim.\n\nThe humor comes from the clever wordplay, as LeBron James, an elite basketball player, might be expected to focus on the game itself, not on the literal act of bringing a ladder. It\'s funny because it takes a common sports expression and

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Lebron James', 'joke': 'Why did LeBron James bring a ladder to the game?\n\nBecause he heard the championship was up for grabs!', 'explanation': 'This joke plays on a couple of puns related to basketball and the concept of "grabs." \n\n1. **Ladder in Basketball**: In basketball, a ladder is often used by players or staff to reach the net for tasks like cutting down the net after winning a championship. This creates a visual image of LeBron James literally bringing a ladder to help him achieve victory.\n\n2. **"Up for Grabs"**: The phrase "up for grabs" is an idiom meaning that something is available to be taken or won. In this context, it refers to the championship being contested, implying that the title is open for any team to claim.\n\nThe humor comes from the clever wordplay, as LeBron James, an elite basketball player, might be expected to focus on the game itself, not on the literal act of bringing a ladder. It\'s funny because it takes a common sp

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Lebron James', 'joke': 'Why did LeBron James bring a ladder to the game?\n\nBecause he heard the championship was up for grabs!', 'explanation': 'This joke plays on a couple of puns related to basketball and the concept of "grabs." \n\n1. **Ladder in Basketball**: In basketball, a ladder is often used by players or staff to reach the net for tasks like cutting down the net after winning a championship. This creates a visual image of LeBron James literally bringing a ladder to help him achieve victory.\n\n2. **"Up for Grabs"**: The phrase "up for grabs" is an idiom meaning that something is available to be taken or won. In this context, it refers to the championship being contested, implying that the title is open for any team to claim.\n\nThe humor comes from the clever wordplay, as LeBron James, an elite basketball player, might be expected to focus on the game itself, not on the literal act of bringing a ladder. It\'s funny because it takes a common s

### Time travel ability in the Langgraph

In [16]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f12e736-2277-6fba-8000-80dcbf6a702a"}})

StateSnapshot(values={'topic': 'Lebron James'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f12e736-2277-6fba-8000-80dcbf6a702a'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-02T09:07:32.806853+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12e736-2273-61bf-bfff-e4f2ae3a4e65'}}, tasks=(PregelTask(id='f9761ae6-5eef-aa09-338d-e430bd407ae3', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did LeBron James bring a ladder to the game?\n\nBecause he heard the championship was up for grabs!'}),), interrupts=())

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f12e731-63e8-652b-bfff-b5b32f5ba580"}})

{'topic': 'pizza',
 'joke': "Why did the pizza maker go broke?\n\nBecause he just couldn't make enough dough! 🍕😄",
 'explanation': 'This joke plays on a clever double meaning of the word "dough." In one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust—this is the literal meaning relevant to the work of a pizza maker. In the other sense, "dough" is a slang term for money. The punchline suggests that the pizza maker went broke because he was unable to earn enough money (dough) to sustain his business, while also making a pun on the fact that he couldn\'t make enough of the pizza crust (dough) either. The humor comes from the play on words and the unexpected twist in the context of a serious situation (going broke).'}

In [19]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go broke?\n\nBecause he just couldn't make enough dough! 🍕😄", 'explanation': 'This joke plays on a clever double meaning of the word "dough." In one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust—this is the literal meaning relevant to the work of a pizza maker. In the other sense, "dough" is a slang term for money. The punchline suggests that the pizza maker went broke because he was unable to earn enough money (dough) to sustain his business, while also making a pun on the fact that he couldn\'t make enough of the pizza crust (dough) either. The humor comes from the play on words and the unexpected twist in the context of a serious situation (going broke).'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12e73d-01f2-6747-8002-b2b6785a2c25'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go broke?\n\nBecause he just couldn't make enough dough! 🍕😄", 'explanation': 'This joke plays on a clever double meaning of the word "dough." In one sense, "dough" refers to the mixture of flour, water, and other ingredients used to make pizza crust—this is the literal meaning relevant to the work of a pizza maker. In the other sense, "dough" is a slang term for money. The punchline suggests that the pizza maker went broke because he was unable to earn enough money (dough) to sustain his business, while also making a pun on the fact that he couldn\'t make enough of the pizza crust (dough) either. The humor comes from the play on words and the unexpected twist in the context of a serious situation (going broke).'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12e73d-01f2-6747-8002-b2b6785a2c25'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='202

### Fault Tolerance : Aritficial Scenario

In [3]:
import time

In [4]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [5]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [6]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume  # RE RUN THIS
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))